# Purchase Data Cleaning

**Purpose:** prepare supplier purchase transactions for cost and purchasing analysis.

**Expected columns:** purchase date, supplier, product/item, quantity, and unit cost.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_FILE = Path('data/purchases_raw.csv')  # Change this if your file has another name
OUTPUT_FILE = Path('output/purchases_cleaned.csv')
OUTPUT_FILE.parent.mkdir(exist_ok=True)

df = pd.read_excel(DATA_FILE) if DATA_FILE.suffix.lower() in {'.xlsx', '.xls'} else pd.read_csv(DATA_FILE)
print(f'Loaded {len(df):,} purchase rows')
df.head()

In [ ]:
# Make column names and text fields consistent
df.columns = (df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('-', '_'))
for column in df.select_dtypes(include='object').columns:
    df[column] = df[column].astype('string').str.strip()
print('Columns:', list(df.columns))

In [ ]:
# Remove duplicate records and convert dates and amounts
rows_before = len(df)
df = df.drop_duplicates().copy()

# Update these names if the columns in your file are different
df['purchase_date'] = pd.to_datetime(df['purchase_date'], errors='coerce')
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df['unit_cost'] = pd.to_numeric(df['unit_cost'], errors='coerce')
df['line_total'] = df['quantity'] * df['unit_cost']

In [ ]:
# Keep valid purchases only
required = ['purchase_date', 'supplier', 'product', 'quantity', 'unit_cost']
df = df.dropna(subset=required)
df = df[(df['quantity'] > 0) & (df['unit_cost'] >= 0)]

audit = pd.DataFrame({
    'measure': ['input rows', 'duplicates removed', 'missing values remaining', 'clean output rows'],
    'value': [rows_before, rows_before - len(df), int(df.isna().sum().sum()), len(df)]
})
display(audit)
df.head()

In [ ]:
df.to_csv(OUTPUT_FILE, index=False)
print(f'Saved cleaned purchase data to: {OUTPUT_FILE}')